## Libraries

In [ ]:
import pandas as pd
import numpy as np

import optuna
from optuna.integration.wandb import WeightsAndBiasesCallback

from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    average_precision_score,
    accuracy_score,
)
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder

import wandb

## Load and Prepare Data

In [50]:
# Load preprocessed datasets
train_df = pd.read_csv("data/preprocessed/train_cleaned.csv")
test_df = pd.read_csv("data/preprocessed/test_cleaned.csv")

In [51]:
train_df.head()

,id,Name,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression,SleepDuration_num,is_student
0,0,Aaradhya,Female,49.0,Ludhiana,Chef,0.0,5.0,0.00,0.0,2.0,Healthy,BHM,No,1.0,2.0,No,0,7.5,0
1,1,Vivan,Male,26.0,Varanasi,Teacher,0.0,4.0,0.00,0.0,3.0,Unhealthy,LLB,Yes,7.0,3.0,No,1,4.5,0
2,2,Yuvraj,Male,33.0,Visakhapatnam,Teacher,5.0,0.0,8.97,2.0,0.0,Healthy,B.Pharm,Yes,3.0,1.0,No,1,5.5,1
3,3,Yuvraj,Male,22.0,Mumbai,Teacher,0.0,5.0,0.00,0.0,1.0,Moderate,BBA,Yes,10.0,1.0,Yes,1,4.5,0
4,4,Rhea,Female,30.0,Kanpur,Business Analyst,0.0,1.0,0.00,0.0,1.0,Unhealthy,BBA,Yes,9.0,4.0,Yes,0,5.5,0


In [52]:
test_df.head()

,id,Name,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,SleepDuration_num,is_student
0,140700,Shivam,Male,53.0,Visakhapatnam,Judge,0.0,2.0,0.00,0.0,5.0,Moderate,LLB,No,9.0,3.0,Yes,4.5,0
1,140701,Sanya,Female,58.0,Kolkata,Educational Consultant,0.0,2.0,0.00,0.0,4.0,Moderate,B.Ed,No,6.0,4.0,No,4.5,0
2,140702,Yash,Male,53.0,Jaipur,Teacher,0.0,4.0,0.00,0.0,1.0,Moderate,B.Arch,Yes,12.0,4.0,No,7.5,0
3,140703,Nalini,Female,23.0,Rajkot,Teacher,5.0,0.0,6.84,1.0,0.0,Moderate,BSc,Yes,10.0,4.0,No,7.5,1
4,140704,Shaurya,Male,47.0,Kalyan,Teacher,0.0,5.0,0.00,0.0,5.0,Moderate,BCA,Yes,3.0,4.0,No,7.5,0


In [53]:
# Prepare data for training
drop_cols = ["id", "Name"]
X = train_df.drop(columns=drop_cols + ["Depression"])
y = train_df["Depression"]
X_test = test_df.drop(columns=drop_cols)

In [54]:
# Identify categorical columns (object type)
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
cat_cols

['Gender',
 'City',
 'Profession',
 'Dietary Habits',
 'Degree',
 'Have you ever had suicidal thoughts ?',
 'Family History of Mental Illness']

In [55]:
# Encode categorical variables
combined = pd.concat([X, X_test], axis=0)

for col in cat_cols:
    le = LabelEncoder()
    # Convert to string to handle potential mixed types
    combined[col] = le.fit_transform(combined[col].astype(str))

In [56]:
# Split back into train and test
X = combined.iloc[: len(X)]
X_test = combined.iloc[len(X) :]

# Split training data for validation (80% train, 20% validation)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=607
)

## Model Training

In [70]:
wandb_kwargs = {
    "entity": "team-csc17001-ida",
    "project": "depression-detection",
    "name": "optuna_xgboost_study2",
}

wandb_callback = WeightsAndBiasesCallback(
    metric_name="valid_f1_macro",  # how the metric will be called in W&B
    wandb_kwargs=wandb_kwargs,  # passed to wandb.init(...)
    as_multirun=False,  # one W&B run for entire study
)

/var/folders/yf/bvb_cdgn46s4l8cp8l3wd_nc0000gn/T/ipykernel_67277/2251763323.py:7: ExperimentalWarning: WeightsAndBiasesCallback is experimental (supported from v2.9.0). The interface can change in the future.
  wandb_callback = WeightsAndBiasesCallback(


In [71]:
@wandb_callback.track_in_wandb()
def objective(trial):
    colsample_bytree = trial.suggest_float("colsample_bytree", 0, 1)
    n_estimators = trial.suggest_int("n_estimators", 400, 1000)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.1)
    reg_lambda = trial.suggest_float("reg_lambda", 0, 4)
    reg_alpha = trial.suggest_float("reg_alpha", 0, 4)
    max_depth = trial.suggest_int("max_depth", 2, 10)
    # num_leaves should be less than or equal to 2^max_depth
    num_leaves = trial.suggest_int("num_leaves", 4, min(256, 2**max_depth))
    gamma = trial.suggest_float("gamma", 0, 0.5)

    # Treat threshold as a hyperparameter
    threshold = trial.suggest_float("threshold", 0.1, 0.9)

    model = XGBClassifier(
        colsample_bytree=colsample_bytree,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=num_leaves,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        gamma=gamma,
        eval_metric="auc",
        random_state=607,
    )
    model.fit(X_train, y_train)

    # Get predicted probabilities
    y_pred_probs = model.predict_proba(X_valid)[:, 1]
    # Apply threshold to get class predictions
    y_pred = (y_pred_probs >= threshold).astype(int)

    roc_auc = roc_auc_score(y_valid, y_pred_probs)
    f1_macro = f1_score(y_valid, y_pred, average="macro")
    average_precision = average_precision_score(y_valid, y_pred_probs)
    accuracy = accuracy_score(y_valid, y_pred)

    # Log to W&B
    wandb.log(
        {
            "valid_auc": roc_auc,
            "valid_ap": average_precision,
            "valid_f1_macro": f1_macro,
            "valid_accuracy": accuracy,
            "threshold": threshold,
        }
    )

    return f1_macro

/var/folders/yf/bvb_cdgn46s4l8cp8l3wd_nc0000gn/T/ipykernel_67277/701190723.py:1: ExperimentalWarning: optuna_integration.wandb.wandb.WeightsAndBiasesCallback.track_in_wandb is experimental (supported from v3.0.0). The interface can change in the future.
  @wandb_callback.track_in_wandb()


In [72]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, callbacks=[wandb_callback], n_jobs=1)
wandb.finish()

[I 2025-11-27 14:16:38,461] A new study created in memory with name: no-name-958595af-1ad1-40f7-8d18-dafd626f59f5
/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/NĂM 4/HKI/Intelligent Data Analysis/Chatbot_Extension/.venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [14:16:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "num_leaves" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2025-11-27 14:16:40,185] Trial 0 finished with value: 0.8900715242139251 and parameters: {'colsample_bytree': 0.16156724626922092, 'n_estimators': 864, 'learning_rate': 0.09772147654212704, 'reg_lambda': 2.5339417980575085, 'reg_alpha': 0.8144349705626945, 'max_depth': 6, 'num_leaves': 37, 'gamma': 0.21401914718433462, 'threshold': 0.6029826561804343}. Best is trial 0 with value: 0.8900715242139251.
/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHC

threshold,▅▆▁▂▆▂▇▆▃▃▄▄▄█▄▅▃▄▄▅▅▅▆▇▅▃▄▄▄▄▄▃▂▁▄▅▅▆▆▄
valid_accuracy,▅▅▁▄▅▄▅▅▅▆▇▇▆▃▆▆▆▅▇█▅█▇▆▆▆▇▇█▅▇▅▅▅███▅▇▇
valid_ap,▃▂▃▂▃▃▃▃▆▆▆▆▄▄█▅▁▇█▆█▇▆▅▇▇▆▇▇▇▆▃▆▇▇██▄▇▆
valid_auc,▃▂▁▃▂▂▃▃▄▆▆▆▆▅▅▆▆▁▇███▆▅▇█▆█▂▇▆▄▆████▄▇▆
valid_f1_macro,▅▄▂▄▄▄▄▅▅▆▇▆▆▁▆▆▆▅▇██▇▆▅▅█▇▇█▅▇▇▅▅███▅▇▇
threshold,0.49198
valid_accuracy,0.96212
valid_ap,0.95746
valid_auc,0.98911
valid_f1_macro,0.93545


In [73]:
best_params = study.best_params
best_score = study.best_value
print(f"Best Hyperparameters: {best_params}")
print(f"Best Accuracy: {best_score:.6f}")

n_estimators = best_params["n_estimators"]
reg_alpha = best_params["reg_alpha"]
learning_rate = best_params["learning_rate"]
reg_lambda = best_params["reg_lambda"]
max_depth = best_params["max_depth"]
num_leaves = best_params["num_leaves"]
colsample_bytree = best_params["colsample_bytree"]
gamma = best_params["gamma"]
threshold = best_params["threshold"]

Best Hyperparameters: {'colsample_bytree': 0.7527796619608157, 'n_estimators': 785, 'learning_rate': 0.06506454595661652, 'reg_lambda': 0.3249164498962406, 'reg_alpha': 2.52176887536818, 'max_depth': 10, 'num_leaves': 256, 'gamma': 0.030639620918430623, 'threshold': 0.5265995390797443}
Best Accuracy: 0.961562


- Best Hyperparameters: `{'colsample_bytree': 0.20685956276418516, 'n_estimators': 754, 'learning_rate': 0.09988004366936079, 'reg_lambda': 2.888255733754247, 'reg_alpha': 0.905208723552474, 'max_depth': 4, 'gamma': 0.13872319644414585, 'threshold': 0.39567845067638424}`
- Best Accuracy: `0.896722`

In [ ]:
# Best params at this moment
params = {
    "colsample_bytree": 0.7527796619608157,
    "n_estimators": 785,
    "learning_rate": 0.06506454595661652,
    "reg_lambda": 0.3249164498962406,
    "reg_alpha": 2.52176887536818,
    "max_depth": 10,
    "num_leaves": 256,
    "gamma": 0.030639620918430623,
    "threshold": 0.5265995390797443,
}

In [75]:
# Define parameters
xgb_model = XGBClassifier(
    colsample_bytree=colsample_bytree,
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    reg_alpha=reg_alpha,
    reg_lambda=reg_lambda,
    max_depth=max_depth,
    num_leaves=num_leaves,
    gamma=gamma,
    eval_metric="auc",
    random_state=607,
)

In [76]:
xgb_model.fit(X_train, y_train)

/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/NĂM 4/HKI/Intelligent Data Analysis/Chatbot_Extension/.venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [14:18:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "num_leaves" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7527796619608157
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'auc'


In [77]:
# Get predicted probabilities
y_pred_probs = xgb_model.predict_proba(X_valid)[:, 1]
# Apply threshold to get class predictions
y_pred = (y_pred_probs >= threshold).astype(int)

In [78]:
# Get confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_valid, y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[22815   246]
 [  387  4692]]


In [79]:
# Get classification report
report = classification_report(y_valid, y_pred)
print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99     23061
           1       0.95      0.92      0.94      5079

    accuracy                           0.98     28140
   macro avg       0.97      0.96      0.96     28140
weighted avg       0.98      0.98      0.98     28140



In [85]:
# Get roc_auc, f1_macro, average_precision, accuracy
roc_auc = roc_auc_score(y_valid, y_pred_probs)
f1_macro = f1_score(y_valid, y_pred, average="macro")
average_precision = average_precision_score(y_valid, y_pred_probs)
accuracy = accuracy_score(y_valid, y_pred)

In [86]:
# Print metrics
print(f"ROC AUC: {roc_auc:.6f}")
print(f"F1 Macro: {f1_macro:.6f}")
print(f"Average Precision: {average_precision:.6f}")
print(f"Accuracy: {accuracy:.6f}")

ROC AUC: 0.994121
F1 Macro: 0.961562
Average Precision: 0.977290
Accuracy: 0.977505


In [80]:
fold_accuracies = []
fold_f1_macros = []
fold_average_precisions = []
fold_roc_aucs = []

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Perform K-Fold Cross Validation
for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    xgb_model.fit(X_train, y_train)

    preds = xgb_model.predict(X_val)

    acc = accuracy_score(y_val, preds)
    f1_macro = f1_score(y_val, preds, average="macro")
    average_precision = average_precision_score(
        y_val, xgb_model.predict_proba(X_val)[:, 1]
    )
    roc_auc = roc_auc_score(y_val, xgb_model.predict_proba(X_val)[:, 1])

    fold_accuracies.append(acc)
    fold_f1_macros.append(f1_macro)
    fold_average_precisions.append(average_precision)
    fold_roc_aucs.append(roc_auc)

    print(f"--- Fold {fold} - Accuracy: {acc:.6f}")
    print(f"--- Fold {fold} - F1-Macro: {f1_macro:.6f}")
    print(f"--- Fold {fold} - Average Precision: {average_precision:.6f}")
    print(f"--- Fold {fold} - ROC-AUC: {roc_auc:.6f}")

# Print overall of each metric
overall_acc = np.mean(fold_accuracies)
overall_f1_macro = np.mean(fold_f1_macros)
overall_average_precision = np.mean(fold_average_precisions)
overall_roc_auc = np.mean(fold_roc_aucs)

print(f"\n------ Overall Accuracy: {overall_acc:.6f}")
print(f"------ Overall F1-Macro: {overall_f1_macro:.6f}")
print(f"------ Overall Average Precision: {overall_average_precision:.6f}")
print(f"------ Overall ROC-AUC: {overall_roc_auc:.6f}")

/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/NĂM 4/HKI/Intelligent Data Analysis/Chatbot_Extension/.venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [14:18:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "num_leaves" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Fold 1 - Accuracy: 0.935039
--- Fold 1 - F1-Macro: 0.889917
--- Fold 1 - Average Precision: 0.897675
--- Fold 1 - ROC-AUC: 0.973618


/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/NĂM 4/HKI/Intelligent Data Analysis/Chatbot_Extension/.venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [14:19:00] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "num_leaves" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Fold 2 - Accuracy: 0.935075
--- Fold 2 - F1-Macro: 0.891107
--- Fold 2 - Average Precision: 0.897059
--- Fold 2 - ROC-AUC: 0.971100


/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/NĂM 4/HKI/Intelligent Data Analysis/Chatbot_Extension/.venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [14:19:04] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "num_leaves" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Fold 3 - Accuracy: 0.935537
--- Fold 3 - F1-Macro: 0.889719
--- Fold 3 - Average Precision: 0.901208
--- Fold 3 - ROC-AUC: 0.971533


/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/NĂM 4/HKI/Intelligent Data Analysis/Chatbot_Extension/.venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [14:19:08] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "num_leaves" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Fold 4 - Accuracy: 0.935572
--- Fold 4 - F1-Macro: 0.889616
--- Fold 4 - Average Precision: 0.900558
--- Fold 4 - ROC-AUC: 0.973877


/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/NĂM 4/HKI/Intelligent Data Analysis/Chatbot_Extension/.venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [14:19:11] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "num_leaves" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Fold 5 - Accuracy: 0.935643
--- Fold 5 - F1-Macro: 0.888003
--- Fold 5 - Average Precision: 0.893805
--- Fold 5 - ROC-AUC: 0.971045

------ Overall Accuracy: 0.935373
------ Overall F1-Macro: 0.889672
------ Overall Average Precision: 0.898061
------ Overall ROC-AUC: 0.972235


In [87]:
# Get predicted probs for test set
test_pred_probs = xgb_model.predict_proba(X_test)[:, 1]
# Apply threshold to get class predictions
test_predictions = (test_pred_probs >= threshold).astype(int)

In [88]:
# Save results to a dataframe
submission = pd.DataFrame({"id": test_df["id"], "Depression": test_predictions})

# Show the first few predictions
print("\nSample Predictions on Test Data:")
print(submission.head())


Sample Predictions on Test Data:
       id  Depression
0  140700           0
1  140701           0
2  140702           0
3  140703           1
4  140704           0


In [89]:
submission.to_csv("xgboost_submission.csv", index=False)